# 01 Data Check & Validation

## Objective
This notebook verifies the dataset structure and checks for issues:
- ✓ Check folder structure
- ✓ Count images per crop and disease
- ✓ Detect empty classes
- ✓ Verify image formats
- ✓ Detect corrupted images
- ✓ Generate dataset summary report

## Dataset Path
Expected structure:
```
Final_images_phd/
├── Brinjal/
├── Castor/
├── Cumin/
├── Guava/
└── Papaya/
```

### Import Libraries

In [13]:
import subprocess
import sys

# Install required packages if missing
packages = ['opencv-python', 'tensorflow', 'scikit-learn']
for package in packages:
    try:
        __import__(package.replace('-', '_'))
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installed")

print("✓ All dependencies checked and installed!")

Installing opencv-python...
✓ opencv-python installed
Installing scikit-learn...
✓ scikit-learn installed
✓ All dependencies checked and installed!


In [14]:
import sys
import os
import cv2
# Add utils to path
sys.path.insert(0, '../utils')

from data_utils import (
    verify_dataset_structure,
    get_image_files,
    find_corrupted_images,
    get_dataset_summary
)

import json
from pathlib import Path
import pandas as pd

print("✓ Libraries imported successfully!")

✓ Libraries imported successfully!


### Set Dataset Path

In [15]:
# Define dataset path
DATASET_PATH = '../Final_images_phd'

# Verify path exists
if os.path.exists(DATASET_PATH):
    print(f"✓ Dataset path found: {DATASET_PATH}")
    print(f"  Absolute path: {os.path.abspath(DATASET_PATH)}")
else:
    print(f"✗ Dataset path NOT found: {DATASET_PATH}")
    print("  Please verify the path and update DATASET_PATH")

✓ Dataset path found: ../Final_images_phd
  Absolute path: c:\Desktop file\2026\cnn_model\Final_images_phd


### 1. Verify Dataset Structure

In [16]:
print("="*70)
print("STEP 1: VERIFYING DATASET STRUCTURE")
print("="*70)

stats = verify_dataset_structure(DATASET_PATH)

# Display results
summary = get_dataset_summary(DATASET_PATH)
print(summary)

STEP 1: VERIFYING DATASET STRUCTURE

DATASET SUMMARY
Total Images: 763
Total Crops: 5

CROP-WISE BREAKDOWN:
------------------------------------------------------------

Brinjal:
  Healthy: 58
  Diseases: 9
    - Cercospora: 17
    - Early_Leaf_spot: 17
    - Leaf_curl: 13
    - Little_leaf: 14
    - Magnessium: 15
    - Mosaic: 8
    - Nitrogen: 14
    - pest_damage: 11
    - Pottasium: 33

Castor:
  Healthy: 35
  Diseases: 6
    - Alternia Leaf Blight: 31
    - Bacterial Leaf Blight: 16
    - Cercospera Leaf Spot: 22
    - Peast_disease: 13
    - Pottassium: 17
    - Wilt: 15

Cumin:
  Healthy: 44
  Diseases: 4
    - Alternatia Blight: 14
    - insect_infection: 12
    - Powdery_Mildew: 12
    - Wilt: 22

Guava:
  Healthy: 51
  Diseases: 8
    - algal_leaf_spot: 16
    - anthrecnose: 15
    - Fungal infection: 15
    - Magnesium: 14
    - Nitrogen: 12
    - phosphorus: 15
    - pottasium: 17
    - Rust_fungal: 11

Papaya:
  Healthy: 35
  Diseases: 8
    - Anthracnose _ Cercospora: 15

### 2. Detailed Crop Statistics

In [17]:
print("\n" + "="*70)
print("STEP 2: DETAILED CROP-WISE STATISTICS")
print("="*70)

# Create detailed table
crop_data = []
for crop, data in stats['crops'].items():
    crop_row = {
        'Crop': crop,
        'Healthy': data['healthy'],
        'Num_Diseases': len(data['unhealthy']),
        'Total_Images': data['healthy'] + sum(data['unhealthy'].values())
    }
    crop_data.append(crop_row)

crops_df = pd.DataFrame(crop_data)
print("\nCrop Summary Table:")
print(crops_df.to_string(index=False))
print(f"\nTotal Images in Dataset: {crops_df['Total_Images'].sum()}")


STEP 2: DETAILED CROP-WISE STATISTICS

Crop Summary Table:
   Crop  Healthy  Num_Diseases  Total_Images
Brinjal       58             9           200
 Castor       35             6           149
  Cumin       44             4           104
  Guava       51             8           166
 Papaya       35             8           144

Total Images in Dataset: 763


### 3. Disease-wise Breakdown

In [18]:


print("\n" + "="*70)
print("STEP 3: DISEASE-WISE IMAGE COUNT")
print("="*70)

disease_data = []
for crop, data in stats['crops'].items():
    # Healthy
    if data['healthy'] > 0:
        disease_data.append({
            'Crop': crop,
            'Disease': 'Healthy',
            'Count': data['healthy']
        })
    
    # Diseases
    for disease, count in data['unhealthy'].items():
        disease_data.append({
            'Crop': crop,
            'Disease': disease,
            'Count': count
        })

disease_df = pd.DataFrame(disease_data)
print("\nDetailed Disease Breakdown:")
print(disease_df.to_string(index=False))
print(f"\nTotal Records: {len(disease_df)}")

# === NEW STEP: Delete images if folder count < 5 ===
base_folder = "dataset"  # change to your dataset path

for _, row in disease_df.iterrows():
    crop = row['Crop']
    disease = row['Disease']
    count = row['Count']
    
    folder_path = os.path.join(base_folder, crop, disease)
    
    if os.path.exists(folder_path):
        images = os.listdir(folder_path)
        if len(images) < 5:
            print(f"Deleting {len(images)} images from {folder_path} (count < 5)")
            for img in images:
                img_path = os.path.join(folder_path, img)
                os.remove(img_path)



STEP 3: DISEASE-WISE IMAGE COUNT

Detailed Disease Breakdown:
   Crop                  Disease  Count
Brinjal                  Healthy     58
Brinjal               Cercospora     17
Brinjal          Early_Leaf_spot     17
Brinjal                Leaf_curl     13
Brinjal              Little_leaf     14
Brinjal               Magnessium     15
Brinjal                   Mosaic      8
Brinjal                 Nitrogen     14
Brinjal              pest_damage     11
Brinjal                Pottasium     33
 Castor                  Healthy     35
 Castor     Alternia Leaf Blight     31
 Castor    Bacterial Leaf Blight     16
 Castor     Cercospera Leaf Spot     22
 Castor            Peast_disease     13
 Castor               Pottassium     17
 Castor                     Wilt     15
  Cumin                  Healthy     44
  Cumin        Alternatia Blight     14
  Cumin         insect_infection     12
  Cumin           Powdery_Mildew     12
  Cumin                     Wilt     22
  Guava          

### 4. Check for Corrupted Images

In [19]:
print("\n" + "="*70)
print("STEP 4: CHECKING FOR CORRUPTED IMAGES")
print("="*70)
print("\nScanning all images...")

corrupted = find_corrupted_images(DATASET_PATH)

if corrupted:
    print(f"\n⚠️  FOUND {len(corrupted)} CORRUPTED IMAGE(S):")
    for idx, img_info in enumerate(corrupted, 1):
        print(f"\n{idx}. {img_info['relative_path']}")
        print(f"   Error: {img_info['error']}")
    
else:
    print(f"\n✓ All images are valid! No corrupted images found.")


STEP 4: CHECKING FOR CORRUPTED IMAGES

Scanning all images...

✓ All images are valid! No corrupted images found.


### 5. Image Format Verification

In [20]:
print("\n" + "="*70)
print("STEP 5: IMAGE FORMAT VERIFICATION")
print("="*70)

from pathlib import Path
from collections import Counter

# Count image formats
format_counter = Counter()
for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif')):
            ext = Path(file).suffix.lower()
            format_counter[ext] += 1

print("\nImage Format Distribution:")
for fmt, count in format_counter.most_common():
    print(f"  {fmt.upper()}: {count} images")
print(f"\nTotal: {sum(format_counter.values())} images")


STEP 5: IMAGE FORMAT VERIFICATION

Image Format Distribution:
  .JPG: 591 images
  .PNG: 130 images
  .JPEG: 49 images

Total: 770 images


### 6. Empty Classes Detection

In [21]:
print("\n" + "="*70)
print("STEP 6: EMPTY CLASSES DETECTION")
print("="*70)

# Find empty classes
empty_classes = []
for crop, data in stats['crops'].items():
    if data['healthy'] == 0:
        empty_classes.append(f"{crop}/Healthy")
    for disease, count in data['unhealthy'].items():
        if count == 0:
            empty_classes.append(f"{crop}/{disease}")

if empty_classes:
    print(f"\n⚠️  FOUND {len(empty_classes)} EMPTY CLASS(ES):")
    for cls in empty_classes:
        print(f"  - {cls}")
else:
    print("\n✓ No empty classes found!")


STEP 6: EMPTY CLASSES DETECTION

✓ No empty classes found!


### 7. Class Balance Analysis

In [22]:
import os
import shutil

print("\n" + "="*70)
print("STEP 7: CLASS BALANCE ANALYSIS")
print("="*70)

# Build class distribution from stats
class_counts = []
for crop, data in stats['crops'].items():
    # Healthy
    if data['healthy'] > 0:
        class_counts.append({
            'Crop': crop,
            'Class': 'Healthy',
            'Count': data['healthy']
        })
    # Diseases
    for disease, count in data['unhealthy'].items():
        class_counts.append({
            'Crop': crop,
            'Class': disease,
            'Count': count
        })

class_df = pd.DataFrame(class_counts)

print("\nClass Distribution:")
print(class_df.to_string(index=False))

# Identify imbalance
min_count = class_df['Count'].min()
max_count = class_df['Count'].max()
avg_count = class_df['Count'].mean()

print("\n⚖️ Balance Summary:")
print(f"  Minimum Class Count: {min_count}")
print(f"  Maximum Class Count: {max_count}")
print(f"  Average Class Count: {avg_count:.2f}")

imbalanced_classes = class_df[class_df['Count'] < 5]
if not imbalanced_classes.empty:
    print("\n⚠️ Classes with < 5 images (imbalanced):")
    print(imbalanced_classes.to_string(index=False))

    # === NEW: Remove entire folder if < 5 images ===
    base_folder = "dataset"  # adjust to your dataset root path
    for _, row in imbalanced_classes.iterrows():
        crop = row['Crop'].strip()
        disease = row['Class'].strip()
        folder_path = os.path.join(base_folder, crop, disease)

        if os.path.exists(folder_path):
            print(f"Removing folder {folder_path} (count < 5)")
            shutil.rmtree(folder_path)  # deletes entire folder
else:
    print("\n✓ No classes with fewer than 5 images.")



STEP 7: CLASS BALANCE ANALYSIS

Class Distribution:
   Crop                    Class  Count
Brinjal                  Healthy     58
Brinjal               Cercospora     17
Brinjal          Early_Leaf_spot     17
Brinjal                Leaf_curl     13
Brinjal              Little_leaf     14
Brinjal               Magnessium     15
Brinjal                   Mosaic      8
Brinjal                 Nitrogen     14
Brinjal              pest_damage     11
Brinjal                Pottasium     33
 Castor                  Healthy     35
 Castor     Alternia Leaf Blight     31
 Castor    Bacterial Leaf Blight     16
 Castor     Cercospera Leaf Spot     22
 Castor            Peast_disease     13
 Castor               Pottassium     17
 Castor                     Wilt     15
  Cumin                  Healthy     44
  Cumin        Alternatia Blight     14
  Cumin         insect_infection     12
  Cumin           Powdery_Mildew     12
  Cumin                     Wilt     22
  Guava                  He

### 8. Generate Data Check Report

In [23]:
print("\n" + "="*70)
print("FINAL REPORT")
print("="*70)

# Build sorted class distribution from Step 7 DataFrame
sorted_classes = (
    class_df[['Crop', 'Class', 'Count']]
    .sort_values(by='Count', ascending=False)
    .to_dict(orient='records')
)

report = {
    'total_images': stats['total_images'],
    'num_crops': len(stats['crops']),
    'total_classes': len(sorted_classes),
    'corrupted_images': len(corrupted),
    'empty_classes': len(empty_classes),
    'crops': list(stats['crops'].keys()),
    'class_distribution': sorted_classes,
    'image_formats': dict(format_counter),
    'issues': stats['issues']
}

print("\n📊 SUMMARY:")
print(f"  Total Images: {report['total_images']}")
print(f"  Total Crops: {report['num_crops']}")
print(f"  Total Classes: {report['total_classes']}")
print(f"  Corrupted Images: {report['corrupted_images']}")
print(f"  Empty Classes: {report['empty_classes']}")
print(f"  Issues Found: {len(report['issues'])}")

# Save report
os.makedirs('../results/reports', exist_ok=True)
report_path = '../results/reports/data_check_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=4)

print(f"\n✓ Report saved to: {report_path}")



FINAL REPORT

📊 SUMMARY:
  Total Images: 763
  Total Crops: 5
  Total Classes: 40
  Corrupted Images: 0
  Empty Classes: 0
  Issues Found: 0

✓ Report saved to: ../results/reports/data_check_report.json


### 9. Status Check

In [24]:
print("\n" + "="*70)
print("✓ DATA VALIDATION COMPLETE")
print("="*70)

print("\n📋 Next Steps:")
print("  1. Review this report")
if len(corrupted) > 0:
    print(f"  2. ⚠️  Fix {len(corrupted)} corrupted image(s)")
if len(empty_classes) > 0:
    print(f"  3. ⚠️  Check {len(empty_classes)} empty class(es)")
print("  4. Proceed to 02_dataset_visualization.ipynb")

print("\n✓ Dataset is ready for preprocessing!" if report['corrupted_images'] == 0 and report['empty_classes'] == 0 else "\n⚠️  Please fix issues before proceeding!")


✓ DATA VALIDATION COMPLETE

📋 Next Steps:
  1. Review this report
  4. Proceed to 02_dataset_visualization.ipynb

✓ Dataset is ready for preprocessing!
